In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install networkx

In [ ]:
import subprocess
import sys
import os

def clean_install():
    """Complete clean installation with correct versions"""
    
    print("🧹 Step 1: Cleaning old packages...")
    subprocess.run([
        sys.executable, "-m", "pip", "uninstall", "-y", 
        "torch", "torchvision", "torchaudio", "transformers",
        "numpy", "scikit-learn", "tensorflow", "keras"
    ], capture_output=True)
    
    print("📦 Step 2: Installing core packages...")
    # Install numpy first with specific version
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy==1.24.3"
    ], check=True)
    
    print("🔥 Step 3: Installing PyTorch...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "torch==2.3.0", "torchvision==0.18.0",
        "--index-url", "https://download.pytorch.org/whl/cu118"
    ], check=True)
    
    print("🤖 Step 4: Installing Transformers...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers==4.44.2"
    ], check=True)
    
    print("📚 Step 5: Installing other dependencies...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "sentence-transformers", "faiss-cpu", 
        "spacy", "networkx", "datasets", "rank_bm25"
    ], check=True)
    
    print("🌐 Step 6: Downloading spacy model...")
    subprocess.run([
        sys.executable, "-m", "spacy", "download", "en_core_web_sm"
    ], capture_output=True)
    
    print("\n" + "="*70)
    print("✅ INSTALLATION COMPLETE!")
    print("="*70)
    print("🔴 IMPORTANT: RESTART RUNTIME NOW!")
    print("="*70)
    print("\nGoogle Colab: Runtime → Restart Runtime")
    print("Jupyter: Kernel → Restart Kernel")
    print("\nAfter restart, run the main code below")
    print("="*70)

# Run installation
clean_install()

In [1]:
%%writefile kaggle_persistence.py
import os
import shutil
import pickle
import numpy as np

def save_kg_to_output(kg_db_path="/kaggle/working/kg_database"):
    """Save full KG state to /kaggle/output for Kaggle dataset creation"""
    output_dir = "/kaggle/output/kg_database"
    os.makedirs(output_dir, exist_ok=True)
    files = [
        "documents.pkl", "titles.pkl", "document_embeddings.npy",
        "knowledge_graph.pkl", "entity_contexts.pkl", "entity_cooccurrence.pkl",
        "community_summaries.pkl", "communities.npy", "metadata.json", "query_history.json"
    ]
    saved = 0
    for f in files:
        src = f"{kg_db_path}/{f}"
        if os.path.exists(src):
            shutil.copy2(src, f"{output_dir}/{f}")
            size_mb = os.path.getsize(f"{output_dir}/{f}") / (1024*1024)
            print(f"  ✅ {f} ({size_mb:.2f} MB)")
            saved += 1
    print(f"\n✅ Saved {saved} files to output")
    
def load_kg_from_dataset(dataset_path="/kaggle/input/scientific-kg-dataset8"):
    """Load pre-built KG from Kaggle dataset"""
    working_dir = "/kaggle/working/kg_database"
    os.makedirs(working_dir, exist_ok=True)
    if not os.path.exists(dataset_path):
        print("❌ Dataset path not found")
        return None
    files = os.listdir(dataset_path)
    for f in files:
        shutil.copy2(f"{dataset_path}/{f}", f"{working_dir}/{f}")
    print(f"✅ Loaded KG from dataset")
    return working_dir

Writing kaggle_persistence.py


In [2]:
import os

import shutil

dataset_path = "/kaggle/input/scientific-kg-dataset8"



if os.path.exists(dataset_path):

    print(f"✅ Dataset found at: {dataset_path}")

    print("Files inside:")

    for f in os.listdir(dataset_path):

        print(f"  - {f}")

else:

    print(f"❌ Dataset NOT found at: {dataset_path}")

    print("➡️ Go to 'Datasets' panel → find your dataset → click 'Add to notebook'")

✅ Dataset found at: /kaggle/input/scientific-kg-dataset8
Files inside:
  - query_history.json
  - knowledge_graph.pkl
  - documents.pkl
  - metadata.json
  - document_embeddings.npy
  - titles.pkl
  - communities.npy
  - entity_contexts.pkl
  - entity_cooccurrence.pkl
  - community_summaries.pkl
  - knowledge_graph_export.json


In [3]:

    dataset_path="/kaggle/input/scientific-kg-dataset8"
    working_dir = "/kaggle/working/kg_database"
    os.makedirs(working_dir, exist_ok=True)
    if not os.path.exists(dataset_path):
        print("❌ Dataset path not found")
        
    files = os.listdir(dataset_path)
    for f in files:
        shutil.copy2(f"{dataset_path}/{f}", f"{working_dir}/{f}")
    print(f"✅ Loaded KG from dataset")

    kg_path=working_dir
    

✅ Loaded KG from dataset


In [4]:
from kaggle_persistence import load_kg_from_dataset

# Load directly from the dataset root
kg_path = load_kg_from_dataset("/kaggle/input/scientific-kg-dataset8")

if kg_path:
    print(f"✅ KG loaded successfully at: {kg_path}")
else:
    print("❌ Failed to load KG")

✅ Loaded KG from dataset
✅ KG loaded successfully at: /kaggle/working/kg_database


In [14]:
#-------------------------------------------------------#
import torch
import faiss
import numpy as np
import spacy
import time
import networkx as nx
import json
import pickle
import os
from datetime import datetime
from sklearn.cluster import KMeans
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from collections import defaultdict

class HIERARCHICAL_LIGHT_RAG:
    def __init__(self, kg_db_path="kg_database"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"🚀 Device: {self.device}")
        # KG Database setup
        self.kg_db_path = f"/kaggle/working/{kg_db_path}"
        os.makedirs(self.kg_db_path, exist_ok=True)
        try:
            self.nlp = spacy.load("en_core_web_sm")
        except Exception as e:
            print(f"⚠️ spaCy loading failed: {e}. Falling back to basic processing.")
            self.nlp = None
        print("🔄 Loading embedding model...")
        self.embedder = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
        print("🔄 Loading generator model...")
        self.tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.generator = AutoModelForSeq2SeqLM.from_pretrained(
            "google/flan-t5-base",
            torch_dtype=dtype,
            device_map="auto" if self.device == "cuda" else None,
            low_cpu_mem_usage=True
        )
        if self.device == "cpu":
            self.generator = self.generator.to(self.device)
        # Core data structures
        self.documents = []
        self.titles = []
        self.document_embeddings = None
        # LIGHTRAG: Hybrid retrieval components
        self.neighbor_index = None
        self.community_index = None
        self.communities = []
        self.community_summaries = {}
        self.community_graph = nx.Graph()
        # Dynamic Knowledge Graph with Local DB
        self.kg = nx.Graph()
        self.entity_contexts = defaultdict(list)
        self.entity_cooccurrence = defaultdict(int)
        self.query_history = []
        # BM25 for sparse retrieval
        self.bm25 = None
        # Load existing KG if available
        self.load_kg_database()
        print("✅ Models loaded and Hybrid RAG initialized")

    def _full_state_exists(self):
        required_files = [
            "documents.pkl", "titles.pkl", "document_embeddings.npy",
            "knowledge_graph.pkl", "entity_contexts.pkl", "entity_cooccurrence.pkl",
            "community_summaries.pkl", "communities.npy", "metadata.json"
        ]
        return all(os.path.exists(f"{self.kg_db_path}/{f}") for f in required_files)

    def save_full_state(self):
        """Save full state for fast reload and Kaggle persistence"""
        print("💾 Saving full state...")
        os.makedirs(self.kg_db_path, exist_ok=True)
        with open(f"{self.kg_db_path}/documents.pkl", "wb") as f:
            pickle.dump(self.documents, f)
        with open(f"{self.kg_db_path}/titles.pkl", "wb") as f:
            pickle.dump(self.titles, f)
        np.save(f"{self.kg_db_path}/document_embeddings.npy", self.document_embeddings)
        with open(f"{self.kg_db_path}/knowledge_graph.pkl", "wb") as f:
            pickle.dump(self.kg, f)
        with open(f"{self.kg_db_path}/entity_contexts.pkl", "wb") as f:
            pickle.dump(dict(self.entity_contexts), f)
        with open(f"{self.kg_db_path}/entity_cooccurrence.pkl", "wb") as f:
            pickle.dump(dict(self.entity_cooccurrence), f)
        with open(f"{self.kg_db_path}/community_summaries.pkl", "wb") as f:
            pickle.dump(self.community_summaries, f)
        with open(f"{self.kg_db_path}/communities.npy", "wb") as f:
            np.save(f, np.array(self.communities))
        with open(f"{self.kg_db_path}/query_history.json", "w") as f:
            json.dump(self.query_history[-100:], f, indent=2)
        metadata = {
            'last_updated': datetime.now().isoformat(),
            'total_entities': self.kg.number_of_nodes(),
            'total_relationships': self.kg.number_of_edges(),
            'total_queries': len(self.query_history),
            'total_documents': len(self.documents)
        }
        with open(f"{self.kg_db_path}/metadata.json", "w") as f:
            json.dump(metadata, f, indent=2)
        print(f"✅ Full state saved: {metadata['total_entities']} entities, {metadata['total_relationships']} relationships")

    def load_full_state(self):
        """Reload full state from disk quickly"""
        print("📂 Loading full state...")
        with open(f"{self.kg_db_path}/documents.pkl", "rb") as f:
            self.documents = pickle.load(f)
        with open(f"{self.kg_db_path}/titles.pkl", "rb") as f:
            self.titles = pickle.load(f)
        self.document_embeddings = np.load(f"{self.kg_db_path}/document_embeddings.npy")
        with open(f"{self.kg_db_path}/knowledge_graph.pkl", "rb") as f:
            self.kg = pickle.load(f)
        with open(f"{self.kg_db_path}/entity_contexts.pkl", "rb") as f:
            self.entity_contexts = defaultdict(list, pickle.load(f))
        with open(f"{self.kg_db_path}/entity_cooccurrence.pkl", "rb") as f:
            self.entity_cooccurrence = defaultdict(int, pickle.load(f))
        with open(f"{self.kg_db_path}/community_summaries.pkl", "rb") as f:
            self.community_summaries = pickle.load(f)
        self.communities = np.load(f"{self.kg_db_path}/communities.npy").tolist()
        if os.path.exists(f"{self.kg_db_path}/query_history.json"):
            with open(f"{self.kg_db_path}/query_history.json", "r") as f:
                self.query_history = json.load(f)
        # Rebuild FAISS index
        dim = self.document_embeddings.shape[1]
        self.neighbor_index = faiss.IndexFlatIP(dim)
        emb_norm = self.document_embeddings.astype(np.float32)
        faiss.normalize_L2(emb_norm)
        self.neighbor_index.add(emb_norm)
        # Rebuild BM25
        tokenized_docs = [doc.lower().split() for doc in self.documents]
        self.bm25 = BM25Okapi(tokenized_docs)
        # Rebuild community index
        centroids = []
        for cid in set(self.communities):
            mask = np.array(self.communities) == cid
            if np.any(mask):
                centroids.append(np.mean(self.document_embeddings[mask], axis=0))
        self.community_index = faiss.IndexFlatIP(dim)
        centroids = np.array(centroids).astype(np.float32)
        faiss.normalize_L2(centroids)
        self.community_index.add(centroids)
        print("✅ Full state reloaded")

    def save_kg_database(self):
        """Save KG and related data to local database"""
        print("💾 Saving KG database...")
        try:
            # Save KG graph
            with open(f"{self.kg_db_path}/knowledge_graph.pkl", 'wb') as f:
                pickle.dump(self.kg, f)
            # Save entity contexts
            with open(f"{self.kg_db_path}/entity_contexts.pkl", 'wb') as f:
                pickle.dump(dict(self.entity_contexts), f)
            # Save entity co-occurrence
            with open(f"{self.kg_db_path}/entity_cooccurrence.pkl", 'wb') as f:
                pickle.dump(dict(self.entity_cooccurrence), f)
            # Save query history as JSON for readability
            with open(f"{self.kg_db_path}/query_history.json", 'w') as f:
                json.dump(self.query_history[-100:], f, indent=2)  # Keep last 100 queries
            # Save community summaries
            with open(f"{self.kg_db_path}/community_summaries.pkl", 'wb') as f:
                pickle.dump(self.community_summaries, f)
            # Save metadata
            metadata = {
                'last_updated': datetime.now().isoformat(),
                'total_entities': self.kg.number_of_nodes(),
                'total_relationships': self.kg.number_of_edges(),
                'total_queries': len(self.query_history),
                'total_documents': len(self.documents)
            }
            with open(f"{self.kg_db_path}/metadata.json", 'w') as f:
                json.dump(metadata, f, indent=2)
            print(f"   ✅ KG Database saved: {metadata['total_entities']} entities, {metadata['total_relationships']} relationships")
        except Exception as e:
            print(f"   ❌ Error saving KG database: {e}")

    def load_kg_database(self):
        """Load KG and related data from local database"""
        print("📂 Loading KG database...")
        try:
            # Load KG graph
            kg_path = f"{self.kg_db_path}/knowledge_graph.pkl"
            if os.path.exists(kg_path):
                with open(kg_path, 'rb') as f:
                    self.kg = pickle.load(f)
                print(f"   ✅ Loaded KG: {self.kg.number_of_nodes()} entities, {self.kg.number_of_edges()} relationships")
            # Load entity contexts
            contexts_path = f"{self.kg_db_path}/entity_contexts.pkl"
            if os.path.exists(contexts_path):
                with open(contexts_path, 'rb') as f:
                    self.entity_contexts.update(pickle.load(f))
                print(f"   ✅ Loaded {len(self.entity_contexts)} entity contexts")
            # Load entity co-occurrence
            cooccur_path = f"{self.kg_db_path}/entity_cooccurrence.pkl"
            if os.path.exists(cooccur_path):
                with open(cooccur_path, 'rb') as f:
                    self.entity_cooccurrence.update(pickle.load(f))
                print(f"   ✅ Loaded {len(self.entity_cooccurrence)} co-occurrence records")
            # Load query history
            history_path = f"{self.kg_db_path}/query_history.json"
            if os.path.exists(history_path):
                with open(history_path, 'r') as f:
                    self.query_history = json.load(f)
                print(f"   ✅ Loaded {len(self.query_history)} query history entries")
            # Load community summaries
            community_path = f"{self.kg_db_path}/community_summaries.pkl"
            if os.path.exists(community_path):
                with open(community_path, 'rb') as f:
                    self.community_summaries = pickle.load(f)
                print(f"   ✅ Loaded {len(self.community_summaries)} community summaries")
            # Load metadata
            metadata_path = f"{self.kg_db_path}/metadata.json"
            if os.path.exists(metadata_path):
                with open(metadata_path, 'r') as f:
                    metadata = json.load(f)
                print(f"   📊 Database stats: {metadata}")
        except Exception as e:
            print(f"   ❌ Error loading KG database: {e}")

    def export_kg_to_json(self, filename="knowledge_graph_export.json"):
        """Export KG to human-readable JSON format"""
        export_data = {
            'entities': {},
            'relationships': [],
            'metadata': {
                'export_date': datetime.now().isoformat(),
                'total_entities': self.kg.number_of_nodes(),
                'total_relationships': self.kg.number_of_edges(),
                'total_queries': len(self.query_history)
            }
        }
        # Export entities with their contexts
        for entity in self.kg.nodes():
            export_data['entities'][entity] = {
                'contexts': self.entity_contexts.get(entity, [])[:5],  # Top 5 contexts
                'degree_centrality': self.kg.degree(entity),
                'neighbors': list(self.kg.neighbors(entity))[:10]  # Top 10 neighbors
            }
        # Export relationships
        for edge in self.kg.edges(data=True):
            export_data['relationships'].append({
                'source': edge[0],
                'target': edge[1],
                'weight': edge[2].get('weight', 1),
                'strength': 'strong' if edge[2].get('weight', 1) > 2 else 'weak'
            })
        # Save to file
        with open(f"{self.kg_db_path}/{filename}", 'w') as f:
            json.dump(export_data, f, indent=2, ensure_ascii=False)
        print(f"📤 KG exported to {filename}")
        return export_data

    def get_kg_statistics(self):
        """Get comprehensive statistics about the knowledge graph"""
        if self.kg.number_of_nodes() == 0:
            return {"message": "Knowledge graph is empty"}
        # Basic stats
        stats = {
            'total_entities': self.kg.number_of_nodes(),
            'total_relationships': self.kg.number_of_edges(),
            'density': nx.density(self.kg),
            'average_degree': sum(dict(self.kg.degree()).values()) / self.kg.number_of_nodes(),
            'connected_components': nx.number_connected_components(self.kg),
            'total_queries': len(self.query_history),
            'total_entity_contexts': sum(len(contexts) for contexts in self.entity_contexts.values())
        }
        # Top entities by degree centrality
        degree_centrality = nx.degree_centrality(self.kg)
        top_entities = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:10]
        stats['top_entities'] = top_entities
        # Strongest relationships
        edges_with_weights = [(u, v, d['weight']) for u, v, d in self.kg.edges(data=True)]
        top_relationships = sorted(edges_with_weights, key=lambda x: x[2], reverse=True)[:10]
        stats['top_relationships'] = top_relationships
        return stats

    def extract_entities(self, text):
        """Extract entities using spaCy or fallback method"""
        entities = set()
        if self.nlp:
            doc = self.nlp(text)
            for ent in doc.ents:
                if ent.label_ in ("PERSON", "GPE", "ORG", "LOC", "NORP", "EVENT", "WORK_OF_ART"):
                    entities.add(ent.text.lower())
            for token in doc:
                if token.pos_ in ("PROPN", "NOUN") and len(token.text) > 3 and token.is_alpha:
                    entities.add(token.text.lower())
        else:
            words = text.lower().split()
            for word in words:
                if len(word) > 3 and word.isalpha():
                    entities.add(word)
        return entities

    def build_knowledge_graph(self, query, retrieved_docs):
        """Build KG dynamically from query and retrieved documents"""
        print("🔄 Building dynamic knowledge graph...")
        # Add to query history
        self.query_history.append({
            'query': query,
            'timestamp': datetime.now().isoformat(),
            'retrieved_docs': len(retrieved_docs)
        })
        # Extract entities from query and documents
        all_entities = set()
        query_entities = self.extract_entities(query)
        all_entities.update(query_entities)
        doc_entities_list = []
        for doc in retrieved_docs[:5]:
            doc_entities = self.extract_entities(doc['content'][:1000])
            all_entities.update(doc_entities)
            doc_entities_list.append(doc_entities)
            # Add document context to entities
            for entity in doc_entities:
                self.entity_contexts[entity].append({
                    'title': doc['title'],
                    'snippet': doc['content'][:200],
                    'source': 'document',
                    'timestamp': datetime.now().isoformat()
                })
        # Add query context
        for entity in query_entities:
            self.entity_contexts[entity].append({
                'title': 'User Query',
                'snippet': query,
                'source': 'query',
                'timestamp': datetime.now().isoformat()
            })
        # Build co-occurrence relationships
        for entities in doc_entities_list:
            entities_list = list(entities)
            for i, ent1 in enumerate(entities_list):
                for ent2 in entities_list[i+1:]:
                    if ent1 != ent2:
                        if self.kg.has_edge(ent1, ent2):
                            self.kg[ent1][ent2]['weight'] += 1
                        else:
                            self.kg.add_edge(ent1, ent2, weight=1)
                        self.entity_cooccurrence[(ent1, ent2)] += 1
        # Add query entity relationships
        query_entities_list = list(query_entities)
        for i, ent1 in enumerate(query_entities_list):
            for ent2 in query_entities_list[i+1:]:
                if ent1 != ent2:
                    if self.kg.has_edge(ent1, ent2):
                        self.kg[ent1][ent2]['weight'] += 2
                    else:
                        self.kg.add_edge(ent1, ent2, weight=2)
                    self.entity_cooccurrence[(ent1, ent2)] += 2
        print(f"   ✅ KG now has {self.kg.number_of_nodes()} entities and {self.kg.number_of_edges()} relationships")
        # Save to database
        self.save_kg_database()

    def find_kg_paths(self, query, max_paths=5, max_depth=2):
        """Find reasoning paths in the knowledge graph"""
        query_entities = self.extract_entities(query)
        all_paths = []
        for entity in query_entities:
            if entity in self.kg:
                for target_entity in self.kg.nodes():
                    if entity != target_entity:
                        try:
                            path = nx.shortest_path(self.kg, source=entity, target=target_entity)
                            if 2 <= len(path) <= max_depth + 1:
                                all_paths.append(path)
                        except nx.NetworkXNoPath:
                            continue
        # Sort paths by strength
        scored_paths = []
        for path in all_paths:
            strength = 0
            for i in range(len(path)-1):
                strength += self.kg[path[i]][path[i+1]]['weight']
            scored_paths.append((path, strength))
        scored_paths.sort(key=lambda x: x[1], reverse=True)
        return [path for path, strength in scored_paths[:max_paths]]

    def get_entity_context(self, entity, max_contexts=3):
        """Get relevant context for an entity"""
        if entity in self.entity_contexts:
            contexts = self.entity_contexts[entity][:max_contexts]
            return [f"{ctx['title']}: {ctx['snippet']}" for ctx in contexts]
        return []

    def build_lightrag_communities(self, n_clusters=50):
        """LIGHTRAG: Build document communities using clustering"""
        print("🔄 Building LIGHTRAG communities...")
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        self.communities = kmeans.fit_predict(self.document_embeddings)
        community_centroids = []
        for i in range(n_clusters):
            cluster_mask = (self.communities == i)
            if np.sum(cluster_mask) > 0:
                centroid = np.mean(self.document_embeddings[cluster_mask], axis=0)
                community_centroids.append(centroid)
        dim = self.document_embeddings.shape[1]
        self.community_index = faiss.IndexFlatIP(dim)
        community_centroids = np.array(community_centroids).astype(np.float32)
        faiss.normalize_L2(community_centroids)
        self.community_index.add(community_centroids)
        print(f"✅ Built {n_clusters} communities")

    def generate_community_summaries(self):
        """RAPTOR: Generate hierarchical summaries for each community"""
        print("🔄 Generating RAPTOR-style community summaries...")
        for community_id in set(self.communities):
            community_docs = [self.documents[i] for i in range(len(self.documents)) 
                            if self.communities[i] == community_id]
            if len(community_docs) > 0:
                community_indices = [i for i in range(len(self.documents)) 
                                   if self.communities[i] == community_id]
                community_embeddings = self.document_embeddings[community_indices]
                centroid = np.mean(community_embeddings, axis=0)
                similarities = np.dot(community_embeddings, centroid)
                top_indices = np.argsort(similarities)[-3:][::-1]
                summary_docs = [community_docs[i] for i in top_indices]
                summary_texts = [doc[:500] for doc in summary_docs[:2]]
                prompt = f"Summarize the main themes and key information from these related documents:\n" + "\n".join(summary_texts)
                inputs = self.tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)
                inputs = {k: v.to(self.device) for k, v in inputs.items()}
                with torch.no_grad():
                    outputs = self.generator.generate(
                        **inputs, max_length=200, num_beams=4, 
                        early_stopping=True, temperature=0.7
                    )
                summary = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                self.community_summaries[community_id] = summary
        print(f"✅ Generated {len(self.community_summaries)} community summaries")
        # Save community summaries to database
        self.save_kg_database()

    def lightrag_hybrid_retrieve(self, query, k=8):
        """LIGHTRAG CORE: Hybrid retrieval (neighbors + communities)"""
        query_embedding = self.embedder.encode([query])
        faiss.normalize_L2(query_embedding)
        neighbor_scores, neighbor_indices = self.neighbor_index.search(
            query_embedding.astype(np.float32), k * 2
        )
        community_scores, community_ids = self.community_index.search(
            query_embedding.astype(np.float32), 5
        )
        community_doc_indices = []
        for community_id in community_ids[0]:
            community_docs = [i for i in range(len(self.documents)) 
                            if self.communities[i] == community_id]
            community_doc_indices.extend(community_docs[:2])
        all_indices = list(neighbor_indices[0]) + community_doc_indices
        unique_indices = list(dict.fromkeys(all_indices))
        relevant_communities = set()
        for idx in unique_indices[:k]:
            relevant_communities.add(self.communities[idx])
        community_context = []
        for comm_id in relevant_communities:
            if comm_id in self.community_summaries:
                community_context.append(f"Community Theme: {self.community_summaries[comm_id]}")
        return unique_indices[:k], community_context

    def load_documents(self, num_docs=3000):
        print(f"📥 Loading {num_docs} documents...")
        dataset = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
        cnt = 0
        for item in dataset:
            if cnt >= num_docs:
                break
            if 'text' not in item or 'title' not in item: 
                continue
            title, text = item['title'], item['text']
            if len(text) < 300: 
                continue
            self.documents.append(text)
            self.titles.append(title)
            cnt += 1
            if cnt % 1000 == 0:
                print(f"   📖 {cnt} documents")
        print(f"✅ Loaded {len(self.documents)} documents")
        self._build_hybrid_indices()

    def _build_hybrid_indices(self):
        """Build both LIGHTRAG and RAPTOR indices"""
        print("🔍 Building hybrid search indices...")
        texts = [f"{t}: {d[:600]}" for t, d in zip(self.titles, self.documents)]
        self.document_embeddings = self.embedder.encode(texts, show_progress_bar=True)
        dim = self.document_embeddings.shape[1]
        self.neighbor_index = faiss.IndexFlatIP(dim)
        embeddings_normalized = self.document_embeddings.astype(np.float32)
        faiss.normalize_L2(embeddings_normalized)
        self.neighbor_index.add(embeddings_normalized)
        self.build_lightrag_communities(n_clusters=min(50, len(self.documents)//10))
        self.generate_community_summaries()
        print("   Building BM25 index...")
        tokenized_docs = [doc.lower().split() for doc in self.documents]
        self.bm25 = BM25Okapi(tokenized_docs)
        print(f"✅ Hybrid indices built with {self.neighbor_index.ntotal} vectors")

    def retrieve(self, query, k=8):
        """Enhanced hybrid retrieval combining LIGHTRAG + RAPTOR"""
        doc_indices, community_context = self.lightrag_hybrid_retrieve(query, k)
        retrieved_docs = [{
            'content': self.documents[idx],
            'title': self.titles[idx],
            'community': self.communities[idx]
        } for idx in doc_indices]
        self.build_knowledge_graph(query, retrieved_docs)
        reasoning_paths = self.find_kg_paths(query)
        enhanced_docs = []
        for idx in doc_indices:
            enhanced_docs.append({
                'content': self.documents[idx],
                'title': self.titles[idx],
                'score': 1.0,
                'community': self.communities[idx]
            })
        return enhanced_docs, community_context, reasoning_paths

    def generate(self, query, docs, community_context, reasoning_paths):
        """Enhanced generation with hybrid context"""
        if not docs:
            return "I couldn't find relevant information to answer this question."
        context_parts = []
        # 1. Document context
        for i, doc in enumerate(docs[:4], 1):
            content = doc['content'][:800].replace('\n', ' ').strip()
            context_parts.append(f"Document {i} ({doc['title']}): {content}")
        # 2. LIGHTRAG community context
        if community_context:
            context_parts.append("THEMATIC CONTEXT:")
            context_parts.extend(community_context[:2])
        # 3. RAPTOR reasoning paths
        if reasoning_paths:
            context_parts.append("KNOWLEDGE PATHS:")
            for i, path in enumerate(reasoning_paths[:3], 1):
                path_str = " → ".join(path)
                context_parts.append(f"Path {i}: {path_str}")
                for entity in path[:2]:
                    entity_contexts = self.get_entity_context(entity, 1)
                    if entity_contexts:
                        context_parts.append(f"   {entity}: {entity_contexts[0][:100]}...")
        prompt = (f"You are a reasoning assistant. Answer the question using the following context.\n"
                 f"CONTEXT:\n{chr(10).join(context_parts)}\n"
                 f"Question: {query}\n"
                 f"Provide a comprehensive answer that synthesizes information from documents, thematic context, and knowledge paths:")
        inputs = self.tokenizer(prompt, return_tensors="pt", max_length=2048, truncation=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.generator.generate(
                **inputs, max_length=400, min_length=100, num_beams=6, length_penalty=1.5,
                early_stopping=True, no_repeat_ngram_size=3, temperature=0.8, do_sample=False
            )
        answer = self.tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        return answer

    def query(self, question, k=8):
        start = time.time()
        print(f"\n❓ {question}")
        docs, community_context, reasoning_paths = self.retrieve(question, k=k)
        print(f"   📚 Retrieved {len(docs)} documents from {len(set([d['community'] for d in docs]))} communities")
        print(f"   🏷️  Community summaries: {len(community_context)}")
        print(f"   🦅 Reasoning paths: {len(reasoning_paths)}")
        for i, doc in enumerate(docs[:3], 1):
            title_preview = doc['title'][:50] + "..." if len(doc['title']) > 50 else doc['title']
            print(f"      {i}. {title_preview} (community: {doc['community']})")
        if community_context:
            print(f"   🎯 Key themes: {', '.join([ctx[:60] + '...' for ctx in community_context[:2]])}")
        if reasoning_paths:
            print(f"   🛣️  Top path: {' → '.join(reasoning_paths[0][:3])}..." if reasoning_paths[0] else "No paths")
        answer = self.generate(question, docs, community_context, reasoning_paths)
        elapsed = time.time() - start
        print(f"   ✅ ANSWER: {answer}")
        print(f"   ⏱️  Time: {elapsed:.2f}s")
        print("-" * 70)
        return {
            'question': question,
            'answer': answer,
            'sources': [d['title'] for d in docs[:4]],
            'community_context': community_context,
            'reasoning_paths': reasoning_paths,
            'kg_stats': {
                'entities': self.kg.number_of_nodes(),
                'relationships': self.kg.number_of_edges()
            },
            'time': elapsed,
            'answer_length': len(answer)
        }

    def show_kg_stats(self):
        """Display comprehensive KG statistics"""
        stats = self.get_kg_statistics()
        print("\n📊 KNOWLEDGE GRAPH STATISTICS")
        print("=" * 50)
        for key, value in stats.items():
            if key not in ['top_entities', 'top_relationships']:
                print(f"   {key.replace('_', ' ').title()}: {value}")
        print(f"\n🏆 Top Entities by Centrality:")
        for entity, centrality in stats.get('top_entities', [])[:5]:
            print(f"   • {entity}: {centrality:.4f}")
        print(f"\n🔗 Strongest Relationships:")
        for source, target, weight in stats.get('top_relationships', [])[:5]:
            print(f"   • {source} ↔ {target} (weight: {weight})")


    def generate_concat(self, query, docs):
        """Ablation: Pure concatenation baseline (no RAPTOR, no KG)"""
        if not docs:
            return "I couldn't find relevant information to answer this question."

        # Concatenate top docs only
        context = "\n\n".join([
            f"Document {i+1} ({doc['title']}): {doc['content'][:800]}"
            for i, doc in enumerate(docs)
        ])

        prompt = (
            f"Answer the question based only on the following context.\n\n"
            f"CONTEXT:\n{context}\n\n"
            f"Question: {query}\n"
            f"Answer:"
        )

       
        inputs = self.tokenizer(
            prompt, 
            return_tensors="pt", 
            max_length=2048, 
            truncation=True,
            padding=True
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.generator.generate(
                **inputs,
                max_length=400,
                min_length=20,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=2,
                temperature=0.7,
                do_sample=False
            )

        answer = self.tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        return answer

    def ablation_query(self, question, k=8, fusion_mode="generate_from_evidence"):
        """
        Run a single query with specified fusion mode for ablation.
        
        fusion_mode: 
          - "concatenate" → simple concatenation
          - "generate_from_evidence" → your full hybrid method
        """
        start = time.time()
        print(f"\n❓ [k={k}, mode={fusion_mode}] {question}")

        if fusion_mode == "concatenate":
            # Only use standard retrieval (no KG, no community, no paths)
            # We'll reuse lightrag_hybrid_retrieve but ignore extra context
            doc_indices, _ = self.lightrag_hybrid_retrieve(question, k=k)
            docs = [{
                'content': self.documents[idx],
                'title': self.titles[idx]
            } for idx in doc_indices[:k]]
            answer = self.generate_concat(question, docs)
            reasoning_paths = []
            community_context = []
        elif fusion_mode == "generate_from_evidence":
            docs, community_context, reasoning_paths = self.retrieve(question, k=k)
            answer = self.generate(question, docs, community_context, reasoning_paths)
        else:
            raise ValueError("Invalid fusion_mode")

        elapsed = time.time() - start
        #print(f"   ✅ ANSWER ({fusion_mode}, k={k}): {answer[:200]}{'...' if len(answer) > 200 else ''}")
        #print(f"   ⏱️  Time: {elapsed:.2f}s")
        print(f"   ✅ ANSWER ({fusion_mode}, k={k}):")
        print(f"      {answer}")
        print(f"   ⏱️  Time: {elapsed:.2f}s")
        print("-" * 70)

        return {
            'question': question,
            'answer': answer,
            'fusion_mode': fusion_mode,
            'k': k,
            'num_docs': len(docs),
            'time': elapsed,
            'answer_length': len(answer)
        }

    def run_ablation_study(self, test_queries=None, k_values=None):
        """
        Run full ablation over k and fusion methods.
        Returns results as list of dicts.
        """
        if test_queries is None:
            test_queries = [
                "What is quantum entanglement?",
                "How does photosynthesis work?",
                "Who was Marie Curie?",
                "Explain the theory of relativity."
            ]
        if k_values is None:
            k_values = [2, 4, 8, 16]

        results = []

        for k in k_values:
            for mode in ["concatenate", "generate_from_evidence"]:
                print(f"\n{'='*50}")
                print(f"📊 ABLATION: k={k}, mode={mode}")
                print(f"{'='*50}")
                for q in test_queries:
                    try:
                        result = self.ablation_query(q, k=k, fusion_mode=mode)
                        results.append(result)
                    except Exception as e:
                        print(f"❌ Error on query '{q}': {e}")
                        results.append({
                            'question': q,
                            'answer': "[ERROR]",
                            'fusion_mode': mode,
                            'k': k,
                            'error': str(e)
                        })

        # Optional: Save results
        with open(f"{self.kg_db_path}/ablation_results.json", "w") as f:
            json.dump(results, f, indent=2)

        print(f"\n✅ Ablation complete! Saved {len(results)} results.")
        return results

In [15]:
def main():

    import numpy as np

    import random

    np.random.seed(42)

    random.seed(42)



    # Initialize RAG system

    rag = HIERARCHICAL_LIGHT_RAG(kg_db_path="kg_database")



    # Load or build knowledge base

    if rag._full_state_exists():

        rag.load_full_state()

        print("✅ Loaded pre-built RAG state.")

    else:

        print("📥 No saved state found. Loading 2000 Wikipedia documents...")

        rag.load_documents(num_docs=2000)

        rag.save_full_state()



    print("="*70)

    print("🧪 RUNNING ABLATION STUDY: Retrieval Size & Fusion Methods")

    print("="*70)



    # Define test queries and k-values for ablation

    test_queries = [

        "What is artificial intelligence?",

        "How does photosynthesis work?",

        "Who was Marie Curie?",

        "Explain the theory of relativity."

    ]

    k_values = [2, 4, 8, 16]



    # Run ablation

    results = rag.run_ablation_study(test_queries=test_queries, k_values=k_values)



    # Optional: Show KG stats at the end

    rag.show_kg_stats()

    

    # Save final KG export for inspection

    rag.export_kg_to_json("ablation_kg_export.json")

    

    print("\n✨ Ablation complete! Results saved to kg_database/ablation_results.json")

    return rag, results


if __name__ == "__main__":

    rag, results = main()

🚀 Device: cuda
🔄 Loading embedding model...


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


🔄 Loading generator model...


/usr/local/lib/python3.11/dist-packages/accelerate/utils/modeling.py:1614: UserWarning: The following device_map keys do not match any submodules in the model: ['decoder.embed_tokens']
  warnings.warn(


📂 Loading KG database...
   ✅ Loaded KG: 1159 entities, 33220 relationships
   ✅ Loaded 1160 entity contexts
   ✅ Loaded 39810 co-occurrence records
   ✅ Loaded 49 query history entries
   ✅ Loaded 50 community summaries
   📊 Database stats: {'last_updated': '2025-12-06T19:04:53.831955', 'total_entities': 1159, 'total_relationships': 33220, 'total_queries': 49, 'total_documents': 2000}
✅ Models loaded and Hybrid RAG initialized
📂 Loading full state...
✅ Full state reloaded
✅ Loaded pre-built RAG state.
🧪 RUNNING ABLATION STUDY: Retrieval Size & Fusion Methods

📊 ABLATION: k=2, mode=concatenate

❓ [k=2, mode=concatenate] What is artificial intelligence?


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


   ✅ ANSWER (concatenate, k=2):
      AI is artificial intelligence, intellectual ability in machines and robots. AI technology is widely used throughout industry, government and science.
   ⏱️  Time: 1.04s
----------------------------------------------------------------------

❓ [k=2, mode=concatenate] How does photosynthesis work?
   ✅ ANSWER (concatenate, k=2):
      ATP is often referred to as the "molecular unit of currency" of intracellular energy transfer
   ⏱️  Time: 0.90s
----------------------------------------------------------------------

❓ [k=2, mode=concatenate] Who was Marie Curie?
   ✅ ANSWER (concatenate, k=2):
      a French surgeon and biologist who was awarded the Nobel Prize in Physiology or Medicine in 1912
   ⏱️  Time: 0.92s
----------------------------------------------------------------------

❓ [k=2, mode=concatenate] Explain the theory of relativity.
   ✅ ANSWER (concatenate, k=2):
      The Big Bang event is a physical theory that describes how the universe

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


   ✅ ANSWER (generate_from_evidence, k=2):
      The intelligence of machines or software, as opposed to the intelligence of humans or animals. It is also the field of study in computer science that develops and studies intelligent machines. "AI" may also refer to the machines themselves. AI technology is widely used throughout industry, government and science. Some high-profile applications are: advanced web search engines (e.g., Google Search), recommendation systems (used by YouTube, Amazon, and Netflix), understanding human speech (such as Siri and Alexa), self-driving cars
   ⏱️  Time: 6.04s
----------------------------------------------------------------------

❓ [k=2, mode=generate_from_evidence] How does photosynthesis work?
🔄 Building dynamic knowledge graph...
   ✅ KG now has 1159 entities and 33220 relationships
💾 Saving KG database...
   ✅ KG Database saved: 1159 entities, 33220 relationships
   ✅ ANSWER (generate_from_evidence, k=2):
      ATP is an organic compound that p

In [16]:
import json
import os
import numpy as np
from collections import defaultdict

RESULTS_FILE = "kg_database/ablation_results.json"

if not os.path.exists(RESULTS_FILE):
    raise FileNotFoundError(f"No ablation results found at {RESULTS_FILE}. Please run the ablation study first.")

with open(RESULTS_FILE, "r") as f:
    ablation_data = json.load(f)

# Group results by configuration
retrieval_by_k = defaultdict(list)
fusion_by_mode = defaultdict(list)

for entry in ablation_data:
    if "error" in entry and entry["error"]:
        continue  # skip failed runs

    # Required fields
    k = entry.get("k")
    mode = entry.get("fusion_mode")
    time_sec = entry.get("time", 0)
    answer_len = entry.get("answer_length", len(entry.get("answer", "")))

    if k is not None:
        retrieval_by_k[k].append({"time": time_sec, "answer_len": answer_len})
    if mode is not None:
        fusion_by_mode[mode].append({"time": time_sec, "answer_len": answer_len})

# ----------------------------
# ABLATION 1: Retrieval Size (k)
# ----------------------------
print("=" * 60)
print("ABLATION 1: RETRIEVAL SIZE (k)")
print("=" * 60)
print(f"{'k':<5} {'Avg Answer Length':<20} {'Avg Time (s)':<15}")
print("-" * 60)

for k in sorted(retrieval_by_k.keys()):
    group = retrieval_by_k[k]
    avg_len = int(np.mean([r["answer_len"] for r in group]))
    avg_time = round(np.mean([r["time"] for r in group]), 2)
    print(f"{k:<5} {avg_len} tokens{'':<12} {avg_time}")

print()


# ----------------------------
# ABLATION 2: Knowledge Graph Impact
# ----------------------------
# Proxy: compare 'concatenate' vs 'generate_from_evidence'
print("=" * 60)
print("ABLATION 3: FUSION STRATEGY (PROXY FOR KG IMPACT)")
print("=" * 60)
print(f"{'Configuration':<25} {'Avg Answer Length':<20} {'Avg Time (s)':<15}")
print("-" * 60)

for mode in sorted(fusion_by_mode.keys()):
    label = "RAG only (concat)" if mode == "concatenate" else "RAG + KG (structured)"
    group = fusion_by_mode[mode]
    avg_len = int(np.mean([r["answer_len"] for r in group]))
    avg_time = round(np.mean([r["time"] for r in group]), 2)
    print(f"{label:<25} {avg_len} tokens{'':<12} {avg_time}")

print("\n💡 Note: 'generate_from_evidence' includes KG paths, community context, and structured prompting.\n")

# ----------------------------
# ABLATION 3: Fusion Methods
# ----------------------------
print("=" * 60)
print("ABLATION 4: FUSION METHODS")
print("=" * 60)
print(f"{'Method':<25} {'Avg Answer Length':<20} {'Avg Time (s)':<15}")
print("-" * 60)

for mode in sorted(fusion_by_mode.keys()):
    display_mode = mode.replace("_", "-").title()
    group = fusion_by_mode[mode]
    avg_len = int(np.mean([r["answer_len"] for r in group]))
    avg_time = round(np.mean([r["time"] for r in group]), 2)
    print(f"{display_mode:<25} {avg_len} tokens{'':<12} {avg_time}")

ABLATION 1: RETRIEVAL SIZE (k)
k     Avg Answer Length    Avg Time (s)   
------------------------------------------------------------
2     291 tokens             3.54
4     273 tokens             5.03
8     266 tokens             5.43
16    243 tokens             5.52

ABLATION 3: FUSION STRATEGY (PROXY FOR KG IMPACT)
Configuration             Avg Answer Length    Avg Time (s)   
------------------------------------------------------------
RAG only (concat)         104 tokens             1.6
RAG + KG (structured)     432 tokens             8.17

💡 Note: 'generate_from_evidence' includes KG paths, community context, and structured prompting.

ABLATION 4: FUSION METHODS
Method                    Avg Answer Length    Avg Time (s)   
------------------------------------------------------------
Concatenate               104 tokens             1.6
Generate-From-Evidence    432 tokens             8.17


In [17]:
components = list(nx.connected_components(rag.kg))
for i, comp in enumerate(components):
    print(f"Component {i+1}: {len(comp)} nodes")
    print(f"  Sample nodes: {list(comp)[:5]}")

Component 1: 1156 nodes
  Sample nodes: ['the art institutes', 'surgeon', 'arabic', "king's college", 'louis']
Component 2: 3 nodes
  Sample nodes: ['marie curie', 'marie', 'curie']
